### Links and Resources
| Function | Use |
|---|---|
| [`transform`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.transform.html) | Apply a lambda to every element |
| [`filter`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.filter.html) | Keep elements matching a predicate |
| [`aggregate`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.aggregate.html) | Reduce / fold an array to a single value |
| [`exists`, `forall`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.exists.html) | Boolean checks |
| [`zip_with`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.zip_with.html) | Pairwise combine two arrays |
| [Higher-order functions blog](https://www.databricks.com/blog/2017/05/24/working-with-nested-data-using-higher-order-functions-in-sql-on-databricks.html) | Background |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Higher-Order Functions on Arrays

A *higher-order function* takes another function — a **lambda** — as input. Spark's higher-order functions let you transform arrays element-by-element **without exploding to rows and re-collecting**. That matters because:

- `explode` → `groupBy` → `collect_list` is expensive (shuffle, aggregation, state).
- A higher-order function is one Catalyst expression that runs on the same row, often vectorized by Photon.

If you've ever written a Python UDF just to map over a list column, this notebook is the replacement.

In [ ]:
df = spark.createDataFrame([
    (1, [10, 20, 30, 40]),
    (2, [5, 15, 25]),
    (3, [100]),
    (4, []),
    (5, None),
], ["id", "vals"])
display(df)

## `transform` — map over elements

Lambda signature: `(element) -> new_element`. Returns a new array of the same length.

In [ ]:
df.select(
    "id", "vals",
    F.transform("vals", lambda x: x * 2).alias("doubled"),
    F.transform("vals", lambda x: F.format_string("v=%d", x)).alias("labeled"),
).show(truncate=False)

### `transform` with index

Pass a 2-arg lambda `(element, index) -> new_element` to get the position. Index is 0-based.

In [ ]:
df.select(
    "id",
    F.transform("vals", lambda x, i: F.struct(i.alias("pos"), x.alias("val"))).alias("indexed"),
).show(truncate=False)

## `filter` — keep elements matching a predicate

The HOF, not `DataFrame.filter`. Lambda returns a Boolean.

In [ ]:
df.select(
    "id", "vals",
    F.filter("vals", lambda x: x > 15).alias("over_15"),
).show(truncate=False)

## `exists` and `forall` — Boolean checks

- `exists(arr, p)` — at least one element satisfies `p`.
- `forall(arr, p)` — every element satisfies `p` (vacuously True for empty arrays).

Cheaper than `array_contains` when the predicate is more than equality, and works on arrays of structs.

In [ ]:
df.select(
    "id", "vals",
    F.exists("vals",  lambda x: x > 50).alias("any_big"),
    F.forall("vals",  lambda x: x > 0).alias("all_positive"),
).show()

## `aggregate` — fold into a single value

Lambda signature: `(accumulator, element) -> new_accumulator`. You provide the **initial value** (which determines the result type) and optionally a **finisher** that runs on the final accumulator.

Think of it as `reduce` with a starting seed.

In [ ]:
df.select(
    "id", "vals",
    F.aggregate("vals", F.lit(0), lambda acc, x: acc + x).alias("sum"),
    F.aggregate("vals", F.lit(1.0), lambda acc, x: acc * x).alias("product"),
).show()

### `aggregate` with a finisher — computing a mean

The finisher converts the working accumulator (here a struct of running sum + count) into the final value (the mean).

In [ ]:
mean_arr = F.aggregate(
    "vals",
    F.struct(F.lit(0.0).alias("s"), F.lit(0).alias("n")),
    lambda acc, x: F.struct((acc.s + x).alias("s"), (acc.n + 1).alias("n")),
    lambda acc: F.when(acc.n > 0, acc.s / acc.n),  # finisher
)

df.select("id", "vals", mean_arr.alias("mean")).show()

## `zip_with` — combine two arrays element-wise

Pairs up elements by index. The result is as long as the **shorter** of the two.

In [ ]:
pairs = spark.createDataFrame([
    (1, [1, 2, 3], [10, 20, 30]),
    (2, [1, 2],    [100, 200, 300]),  # extra element in second array is dropped
], ["id", "a", "b"])

pairs.select(
    "id",
    F.zip_with("a", "b", lambda x, y: x + y).alias("sum"),
    F.zip_with("a", "b", lambda x, y: F.struct(x.alias("a"), y.alias("b"))).alias("paired"),
).show(truncate=False)

## SQL syntax — same functions, lambda arrows

In SQL you write `x -> ...` instead of a Python `lambda`. Same semantics, often cleaner inside a long query.

In [ ]:
df.createOrReplaceTempView("v")

spark.sql("""
SELECT id,
       transform(vals, x -> x * 2)              AS doubled,
       filter(vals,    x -> x > 15)             AS over_15,
       aggregate(vals, 0, (acc, x) -> acc + x)  AS sum
FROM v
""").show(truncate=False)

## Why HOFs beat the explode→groupBy pattern

Two equivalent ways to sum each row's array:

```python
# Old way — shuffle and rebuild
(df.select("id", F.explode("vals").alias("v"))
   .groupBy("id").agg(F.sum("v").alias("sum")))

# HOF way — one expression, no shuffle, stays per-row
df.select("id", F.aggregate("vals", F.lit(0), lambda a, x: a + x).alias("sum"))
```

The HOF version:
- Stays inside the row — no shuffle, no extra stage.
- Fuses with surrounding projections.
- Photon can JIT it in C++.

Use the explode pattern only when you genuinely need rows (e.g. to join against another table).

## NULL semantics

- HOFs on a NULL array return NULL. Empty array stays empty.
- The lambda *body* can return NULL — those NULLs end up in the result array unchanged.
- `forall` on an empty array is **True** (vacuously). `exists` on empty is **False**.

## Takeaways

- Use `transform` / `filter` instead of explode-and-rebuild for array element work.
- `aggregate(arr, init, merge_fn, finisher_fn)` is `reduce` with an optional finisher — perfect for sums, products, means, counts.
- `exists` / `forall` are the `any` / `all` of arrays.
- `zip_with` pairs two arrays by index; result length = min(len_a, len_b).
- In SQL, lambdas use `->` syntax; everything else is identical.